## Experimenting DkNN

In [ ]:
# python stuff

from pathlib import Path as Path
from numpy.random import randint

# Our stuff
from datasets.cifar import Cifar
from models.model_wrap import ModelWrap

# from credibility import get_credibility

# torch stuff
import torch
from torchvision.models import vgg16, VGG16_Weights
from peepholes.peepholes import Peepholes
from peepholes.svd_peepholes import peep_matrices_from_svds as parser_fn
from credibility.DkNN import NearestNeighbor, DkNN


use_cuda = torch.cuda.is_available()
cuda_index = torch.cuda.device_count() - 3
device = torch.device(f"cuda:{cuda_index}" if use_cuda else "cpu")
print(f"Using {device} device")

#--------------------------------
# Dataset 
#--------------------------------
# model parameters
dataset = 'CIFAR100' 
seed = 29
bs = 64
data_path = '/srv/newpenny/XAI/LM/data/CIFAR100'

ds = Cifar(dataset=dataset, data_path=data_path)
ds.load_data(
        batch_size = bs,
        data_kwargs = {'num_workers': 4, 'pin_memory': True},
        seed = seed,
        )

In [ ]:
#--------------------------------
# Model 
#--------------------------------
pretrained = True
model_dir = '/srv/newpenny/XAI/LM/models'
model_name = f'vgg16_pretrained={pretrained}_dataset={dataset}-'\
f'augmented_policy=CIFAR10_bs={bs}_seed={seed}.pth'

nn = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
in_features = 4096
num_classes = len(ds.get_classes()) 
nn.classifier[-1] = torch.nn.Linear(in_features, num_classes)
model = ModelWrap(device=device)
model.set_model(model=nn, path=model_dir, name=model_name, verbose=True)

layers_dict = {'classifier': [0]}
              
model.set_target_layers(target_layers=layers_dict, verbose=True)
print('target layers: ', model.get_target_layers()) 

direction = {'save_input':True, 'save_output':False}
model.add_hooks(**direction, verbose=False) 

dry_img, _ = ds._train_ds.dataset[0]
dry_img = dry_img.reshape((1,)+dry_img.shape)
model.dry_run(x=dry_img)

#--------------------------------
# SVDs 
#--------------------------------
svds_path = Path.cwd()/'../data/svdsBanana'
svds_name = 'svdsBatata' 
model.get_svds(model=model, path=svds_path, name=svds_name, verbose=True)
for k in model._svds.keys():
    print('svd shapes: ', k, model._svds[k]['Vh'].shape)
#--------------------------------
# Peepholes 
#--------------------------------
phs_name = 'peepholes'
phs_dir = Path.cwd()/'../data/peepholes'
peepholes = Peepholes(
        path = phs_dir,
        name = phs_name,
        )
loaders = ds.get_dataset_loaders()
# copy dataset to peepholes dataset
peepholes.get_peep_dataset(
        loaders = loaders,
        verbose = True
        ) 

peepholes.get_activations(
        model=model,
        loaders=loaders,
        verbose=True
        )

peepholes.get_peepholes(
        model = model,
        peep_matrices = model._svds,
        parser = parser_fn,
        verbose = True
        )

In [ ]:
def get_act_from_ds(peephole, model, portion):
    """
    Extract the activations specific for a set of layer and the labels for a specific dataset
    """
    dataloader = peephole._loaders[portion]
    if dataloader.batch_size == len(dataloader.dataset):
        activations = {}
        for data in dataloader:
            labels = data['label'].detach().cpu().numpy().astype(int)
            for key,layer in model._target_layers.items():
                if isinstance(layer, torch.nn.Linear):
                    activations[key] = data['in_activations'][key].detach().cpu().numpy()
                if isinstance(layer, torch.nn.Conv2d):
                    flatten_act = data['in_activations'][key].flatten(start_dim=1)
                    activations[key] = flatten_act.detach().cpu().numpy()
    else:
        raise RuntimeError('set DataLoader batch_size equal to the length of the dataset')

    return activations, labels 

In [146]:
portion = 'train'

In [87]:
activations, labels = get_act_from_ds(peephole=peepholes, model=model, portion=portion)

In [88]:
import numpy as np

In [37]:
labels_hist = np.bincount(labels)
labels_hist

array([400, 389, 400, 429, 403, 384, 393, 393, 394, 396, 405, 415, 389,
       394, 403, 410, 419, 401, 409, 414, 386, 424, 397, 378, 406, 406,
       404, 395, 384, 402, 395, 393, 375, 392, 389, 393, 383, 402, 407,
       389, 419, 387, 405, 392, 382, 397, 402, 381, 389, 403, 392, 406,
       394, 413, 395, 410, 394, 395, 387, 403, 407, 390, 386, 422, 388,
       410, 404, 388, 403, 398, 412, 404, 390, 414, 410, 394, 403, 416,
       398, 397, 399, 407, 408, 416, 406, 404, 396, 402, 406, 401, 388,
       402, 407, 419, 407, 403, 397, 400, 402, 410])

In [48]:
classes = np.arange(0,nb_classes)
classes

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

In [89]:
percentage[portion]

5

In [140]:
n_samples = len(labels)
rng = np.random.default_rng(seed)
idx_rand = []

for l in np.arange(0,nb_classes):
    idx_l = np.argwhere(labels==l)
    n_samples_l = len(idx_l)
    num_elements_to_select = int(n_samples_l*(percentage[portion]/100))
    rng.shuffle(idx_l)
    idx_rand.append(idx_l[:num_elements_to_select])

idx_rand = np.concatenate(idx_rand)

In [103]:
num_elements_to_select = np.ceil(n_samples_l*(percentage[portion]/100))
num_elements_to_select

np.float64(20.0)

In [135]:
labels.shape, idx_rand.shape, int(n_samples*(percentage[portion]/100))

((40000,), (39646, 1), 39600)

In [138]:
labels.shape, idx_rand.shape, int(n_samples*(percentage[portion]/100))

((40000,), (39549, 1), 39600)

In [141]:
np.bincount(labels[idx_rand[:,0]]), np.bincount(labels)

(array([20, 19, 20, 21, 20, 19, 19, 19, 19, 19, 20, 20, 19, 19, 20, 20, 20,
        20, 20, 20, 19, 21, 19, 18, 20, 20, 20, 19, 19, 20, 19, 19, 18, 19,
        19, 19, 19, 20, 20, 19, 20, 19, 20, 19, 19, 19, 20, 19, 19, 20, 19,
        20, 19, 20, 19, 20, 19, 19, 19, 20, 20, 19, 19, 21, 19, 20, 20, 19,
        20, 19, 20, 20, 19, 20, 20, 19, 20, 20, 19, 19, 19, 20, 20, 20, 20,
        20, 19, 20, 20, 20, 19, 20, 20, 20, 20, 20, 19, 20, 20, 20]),
 array([400, 389, 400, 429, 403, 384, 393, 393, 394, 396, 405, 415, 389,
        394, 403, 410, 419, 401, 409, 414, 386, 424, 397, 378, 406, 406,
        404, 395, 384, 402, 395, 393, 375, 392, 389, 393, 383, 402, 407,
        389, 419, 387, 405, 392, 382, 397, 402, 381, 389, 403, 392, 406,
        394, 413, 395, 410, 394, 395, 387, 403, 407, 390, 386, 422, 388,
        410, 404, 388, 403, 398, 412, 404, 390, 414, 410, 394, 403, 416,
        398, 397, 399, 407, 408, 416, 406, 404, 396, 402, 406, 401, 388,
        402, 407, 419, 407, 403, 397, 4

In [142]:
idx = 0
labels_hist*percentage[portion]/100

array([20.  , 19.45, 20.  , 21.45, 20.15, 19.2 , 19.65, 19.65, 19.7 ,
       19.8 , 20.25, 20.75, 19.45, 19.7 , 20.15, 20.5 , 20.95, 20.05,
       20.45, 20.7 , 19.3 , 21.2 , 19.85, 18.9 , 20.3 , 20.3 , 20.2 ,
       19.75, 19.2 , 20.1 , 19.75, 19.65, 18.75, 19.6 , 19.45, 19.65,
       19.15, 20.1 , 20.35, 19.45, 20.95, 19.35, 20.25, 19.6 , 19.1 ,
       19.85, 20.1 , 19.05, 19.45, 20.15, 19.6 , 20.3 , 19.7 , 20.65,
       19.75, 20.5 , 19.7 , 19.75, 19.35, 20.15, 20.35, 19.5 , 19.3 ,
       21.1 , 19.4 , 20.5 , 20.2 , 19.4 , 20.15, 19.9 , 20.6 , 20.2 ,
       19.5 , 20.7 , 20.5 , 19.7 , 20.15, 20.8 , 19.9 , 19.85, 19.95,
       20.35, 20.4 , 20.8 , 20.3 , 20.2 , 19.8 , 20.1 , 20.3 , 20.05,
       19.4 , 20.1 , 20.35, 20.95, 20.35, 20.15, 19.85, 20.  , 20.1 ,
       20.5 ])

In [ ]:
ds.config

In [ ]:
peepholes._peepds

In [ ]:
batch_dict = {key : value for key, value in peepholes._n_samples.items()}
kwargs = {'batch_dict': batch_dict,
          'verbose': True}
ph_dl = peepholes.get_dataloaders(**kwargs)

In [ ]:
peepholes._loaders


In [7]:
peepholes._peepds

{'train': TensorDict(
     fields={
         image: MemoryMappedTensor(shape=torch.Size([40000, 3, 224, 224]), device=cpu, dtype=torch.float32, is_shared=True),
         in_activations: TensorDict(
             fields={
                 classifier.0: MemoryMappedTensor(shape=torch.Size([40000, 25088]), device=cpu, dtype=torch.float32, is_shared=True),
                 classifier.3: MemoryMappedTensor(shape=torch.Size([40000, 4096]), device=cpu, dtype=torch.float32, is_shared=True),
                 features.28: MemoryMappedTensor(shape=torch.Size([40000, 512, 14, 14]), device=cpu, dtype=torch.float32, is_shared=True)},
             batch_size=torch.Size([40000]),
             device=cpu,
             is_shared=False),
         label: MemoryMappedTensor(shape=torch.Size([40000]), device=cpu, dtype=torch.float32, is_shared=True),
         peepholes: TensorDict(
             fields={
                 classifier.0: MemoryMappedTensor(shape=torch.Size([40000, 4096]), device=cpu, dtype=torch

In [8]:
peepholes._n_samples, batch_dict

({'train': 40000, 'val': 10000, 'test': 10000},
 {'train': 40000, 'val': 10000, 'test': 10000})

# Initialize DkNN

In [139]:
neighbors = 75 
percentage = {'train':5,
               'val':1,
               'test':1}
nb_classes = ds.config['num_classes']
layers = model._target_layers
verbose = True

In [10]:
dknn = DkNN(model, 
            nb_classes, 
            neighbors, 
            layers, 
            peepholes, 
            percentage, 
            seed, 
            verbose,
            NearestNeighbor.BACKEND.FALCONN)

---------- DkNN init

## Constructing the NearestNeighbor tables
Constructing table for classifier.0
done!



In [11]:
dknn.peephole._loaders

{'train': <torch.utils.data.dataloader.DataLoader at 0x7f4aad08a750>,
 'val': <torch.utils.data.dataloader.DataLoader at 0x7f4ab59bd640>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7f4aa50ffbc0>}

In [12]:
peepholes._loaders

{'train': <torch.utils.data.dataloader.DataLoader at 0x7f4aad08a750>,
 'val': <torch.utils.data.dataloader.DataLoader at 0x7f4ab59bd640>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7f4aa50ffbc0>}

In [13]:
list(model._target_layers.keys()), list(percentage.values())[:-1]

(['classifier.0'], [1, 1])

In [14]:
dknn.calibrate()

---------- DkNN calibrate

## Starting calibration of DkNN
DkNN calibration complete


In [15]:
preds_knn, confs, creds, p_value, test_labels = dknn.fprop('test')

---------- DkNN predict

Nonconformity calculated
Predictions created


In [16]:
torch.Tensor(confs)

tensor([0.0707, 0.0707, 0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101,
        0.0707, 0.0707, 0.0707, 0.0101, 0.0101, 0.0707, 0.0101, 0.0707, 0.0707,
        0.0707, 0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101,
        0.0707, 0.0707, 0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101,
        0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101, 0.0101, 0.0707,
        0.0101, 0.0101, 0.0707, 0.0101, 0.0101, 0.0101, 0.0707, 0.0707, 0.0101,
        0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707,
        0.0101, 0.0707, 0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707,
        0.0707, 0.0101, 0.0707, 0.0101, 0.0707, 0.0707, 0.0101, 0.0101, 0.0707,
        0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101, 0.0707, 0.0707, 0.0707,
        0.0707, 0.0101, 0.0707, 0.0101, 0.0101, 0.0707, 0.0101, 0.0707, 0.0707,
        0.0707])

In [17]:
peepholes._file_paths

{'train': PosixPath('/home/lorenzocapelli/repos/XAI/src/../data/peepholes/peepholes.train'),
 'val': PosixPath('/home/lorenzocapelli/repos/XAI/src/../data/peepholes/peepholes.val'),
 'test': PosixPath('/home/lorenzocapelli/repos/XAI/src/../data/peepholes/peepholes.test')}

In [18]:
preds_knn, confs, creds, p_value, test_labels = dknn.fprop('train')

---------- DkNN predict

Nonconformity calculated
Predictions created


In [19]:
confs.shape, p_value.shape

((400,), (400, 100))

In [20]:
peepholes._n_samples

{'train': 40000, 'val': 10000, 'test': 10000}

In [21]:
dknn.fprop(portion='all')

---------- DkNN predict


 ---- Getting activations for train

40000
adding scores tensorDict
adding DkNN tensorDict
Nonconformity calculated


RuntimeError: batch dimension mismatch, got self.batch_size=torch.Size([2]) and value.shape=torch.Size([40000]).

In [23]:
peepds.keys()

dict_keys(['train', 'val', 'test'])

In [27]:
peepds['train']

In [36]:
confs

NameError: name 'confs' is not defined

In [ ]:
'''
def DkNN_scores():
    
    def __init__(self, **kwargs):
        self.path = Path(kwargs['path'])
        self.name = Path(kwargs['name'])
        # create folder
        self.path.mkdir(parents=True, exist_ok=True)

        # computed in get_peep_dataset()
        self._file_paths = None
        return
'''  
def load_scores(self, **kwargs):
    path = Path(kwargs['path'])
    name = Path(kwargs['name'])
    # create folder
    portion = ['train', 'val', 'test']
    path.mkdir(parents=True, exist_ok=True)

    # computed in get_peep_dataset()
    _file_paths = None

    _scores = {}
    
    verbose = kwargs['verbose'] if 'verbose' in kwargs else False
    for p in poriton:
        if verbose: print(f'\n ---- Loading results from {p}\n')
        file_path = self.path/(self.name.name+'.'+p)
        _file_paths[p] = file_path

        if file_path.exists():
            if verbose: print(f'File {file_path} exists. Loading from disk.')
            _scores[p] = TensorDict.load_memmap(file_path)
        else:
            raise RuntimeError(f'File {file_path} does not exist.') 
    return _scores

def store_scores(self, **kwargs):
    path = Path(kwargs['path'])
    name = Path(kwargs['name'])
    if file_path.exists():
        if 